# Building Footprint Extraction - India

Detects building footprints from a HOTOSM/OIN aerial/satellite image over India, using the `geoai-py` package (following the [Africa building footprints example](https://opengeoai.org/examples/building_footprints_africa/)).

The raster is too large to process in a single pass, so it's split into georeferenced tiles first (following the [image tiling example](https://opengeoai.org/examples/image_tiling/)), the extractor is run on each tile, and the resulting masks are mosaicked back together.

In [ ]:
%pip install -q geoai-py rioxarray

## Import libraries

In [ ]:
import glob
import os
import shutil
from urllib.parse import urlparse

import geoai
import rasterio
import rioxarray
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform

## Download sample data

In [ ]:
image1 = 'https://oin-hotosm-temp.s3.us-east-1.amazonaws.com/69493c8084a859b011c94266/0/69493c8084a859b011c94267.tif' # Raichur Jaltol
image2 = 'https://oin-hotosm-temp.s3.us-east-1.amazonaws.com/66de82e8cd0baa0001b61fe7/0/66de82e8cd0baa0001b61fe8.tif' # Singhnagar floods
raster_url = image1

## Clear previous run's outputs

Remove tiles, tile masks, and the previously downloaded raster so re-running the notebook starts from a clean state.

In [ ]:
tiles_dir = "tiles"
masks_dir = "tile_masks"
downloaded_path = os.path.basename(urlparse(raster_url).path)

shutil.rmtree(tiles_dir, ignore_errors=True)
shutil.rmtree(masks_dir, ignore_errors=True)
if os.path.exists(downloaded_path):
    os.remove(downloaded_path)

In [ ]:
raster_path = geoai.download_file(raster_url)

## Initialize the model

In [ ]:
extractor = geoai.BuildingFootprintExtractor(model_path="building_footprints_usa.pth")

## Split raster into tiles

In [ ]:
tiles_dir = "tiles"
masks_dir = "tile_masks"

geoai.export_geotiff_tiles(
    in_raster=raster_path,
    out_folder=tiles_dir,
    tile_size=10000,
    stride=10000,
)

## Extract building footprints per tile

In [ ]:
target_resolution = 0.5  # meters (50 cm)


def native_resolution_m(tile_path):
    """Native pixel size in meters, without reprojecting the actual data."""
    with rasterio.open(tile_path) as src:
        if not src.crs.is_geographic:
            return abs(src.transform.a)
        crs, bounds, width, height = src.crs, src.bounds, src.width, src.height
    utm_crs = rioxarray.open_rasterio(tile_path).rio.estimate_utm_crs()
    transform, _, _ = calculate_default_transform(crs, utm_crs, width, height, *bounds)
    return abs(transform.a)


os.makedirs(masks_dir, exist_ok=True)
tile_paths = sorted(glob.glob(f"{tiles_dir}/images/*.tif"))
print(f"Tiles to process: {len(tile_paths)}")

for tile_path in tile_paths:
    if native_resolution_m(tile_path) > target_resolution:
        # Already coarser than target_resolution: skip reprojection/resampling.
        # Tiles still inherit the source raster's YCBCR/JPEG photometric tag,
        # which conflicts with generate_masks' LZW-compressed output profile,
        # so rewrite with a clean profile.
        with rasterio.open(tile_path) as src:
            data = src.read()
            profile = src.profile.copy()
        profile.pop("photometric", None)
        profile.update(compress="deflate")
        with rasterio.open(tile_path, "w", **profile) as dst:
            dst.write(data)
    else:
        # Finer than target_resolution: reproject to local UTM and resample
        # to target_resolution. Done per tile (not on the full raster) to
        # keep memory use bounded.
        tile_da = rioxarray.open_rasterio(tile_path)
        utm_crs = tile_da.rio.estimate_utm_crs()
        tile_da_utm = tile_da.rio.reproject(
            utm_crs,
            resolution=target_resolution,
            resampling=Resampling.bilinear,
            nodata=0,
        )
        tile_da_utm.rio.to_raster(tile_path, compress="deflate")
        tile_da.close()
        tile_da_utm.close()

    out_path = os.path.join(masks_dir, os.path.basename(tile_path))
    print(f"Processing {tile_path}...")

    extractor.generate_masks(
        tile_path,
        output_path=out_path,
        min_object_area=10,
        confidence_threshold=0.5,
        mask_threshold=0.5,
    )

## Mosaic tile masks

In [ ]:
masks_path = "building_masks.tif"
geoai.mosaic_geotiffs(input_dir=masks_dir, output_file=masks_path)

## Vectorize masks and regularize building footprints

In [ ]:
gdf = geoai.orthogonalize(
    input_path=masks_path, output_path="building_footprints.geojson", epsilon=1.0
)
gdf = geoai.regularization(
    building_polygons=gdf,
    angle_tolerance=0,
    simplify_tolerance=0.5,
    orthogonalize=True,
    preserve_topology=True,
)
gdf = geoai.add_geometric_properties(gdf)

## Visualize results

In [14]:
geoai.view_vector_interactive(
    gdf, style_kwds={"color": "red", "fillOpacity": 0.2}, tiles=raster_url
)

## Export Results

In [13]:
gdf.to_crs('EPSG:4326').to_file("building_footprints.geojson", driver="GeoJSON")